In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.ccny.cuny.edu/registrar/fall'
response = requests.get(url)
html_doc = response.text

soup = BeautifulSoup(html_doc, 'html.parser')

# Fall back to saved file if Cloudflare blocked
# Have ccny calendar as a html file
if not soup.find('table'):
    with open('fall.html', 'r', encoding='utf-8') as f:
        html_doc = f.read()
    soup = BeautifulSoup(html_doc, 'html.parser')

# Calendar is located at table 4
table = soup.find_all('table')[4]

# Extract rows
rows = []
for tr in table.find_all('tr'):
    cells = tr.find_all('td')
    if not cells:
        continue
    rows.append([td.text.strip() for td in cells])

df = pd.DataFrame(rows, columns=['date', 'dow', 'text'])

df['date'] = pd.to_datetime(df['date'] + ', 2021', format='%B %d, %Y', errors='coerce')
df = df.dropna(subset=['date']).set_index('date')

# Clean whitespace in text
df['text'] = df['text'].str.replace(r'[\n\t]+', ' ', regex=True)

print(df)

                  dow                                               text
date                                                                    
2021-08-01     Sunday  Application for degree for January and Februar...
2021-08-18  Wednesday                 Last day to apply for Study Abroad
2021-08-24    Tuesday  Last day of Registration; Last day to file ePe...
2021-08-25  Wednesday  Start of Fall Term; Classes begin; Initial Reg...
2021-08-26   Thursday                     Last day for Independent Study
2021-08-28   Saturday                      First day of Saturday Classes
2021-08-31    Tuesday  Last day to add a class to an existing enrollm...
2021-09-01  Wednesday  Verification of Enrollment rosters available t...
2021-09-06     Monday   College Closed; Last day for 50% tuition refund;
2021-09-09   Thursday                               Freshman Convocation
2021-09-14    Tuesday  Last day for 25% tuition refund; Census date; ...
2021-09-15  Wednesday  Assignment of 'WN' grades fo